# 03 - Data Cleaning: Drop, Impute, and Encode

**Milestone 1 — Part 3 (A–E)**: Systematically clean the dataset.

## Objectives
- **3.A** Drop features unsuitable for regression (business logic)
- **3.B** Drop features with too many null values
- **3.C** Drop problematic samples (too many nulls, null target, outliers)
- **3.D** Impute remaining missing values
- **3.E** Encode categorical features

## Expected Outcomes
| Deliverable | Description |
|---|---|
| `df_dropped_features` | DataFrame after removing useless features |
| `df_dropped_nulls` | After dropping high-null features |
| `df_dropped_samples` | After removing problematic rows |
| `df_imputed` | After imputing remaining NaNs — zero nulls |
| `df_encoded` | After encoding categoricals — ready for modeling |
| Reusable functions | Each step wrapped in a function for reproducibility |

## Key Rule (from assignment)
> Create new variable names at each stage. Write functions for all transformations.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
TARGET = "taxvaluedollarcnt"

df = pd.read_csv("zillow_dataset.csv")
print(f"Starting shape: {df.shape}")

## 3.A — Drop Features Unsuitable for Regression

Drop columns that have no predictive value based on domain reasoning:
- **Identifiers**: `parcelid` — unique ID, not a predictor
- **Redundant location codes**: e.g., `rawcensustractandblock`, `censustractandblock`
- **Assessment metadata**: `assessmentyear` if all same year

TODO: Refine this list after reviewing output from notebook 01.

In [ ]:
def drop_useless_features(dataframe: pd.DataFrame, columns_to_drop: list) -> pd.DataFrame:
    """Drop columns that are not useful for the regression task."""
    existing = [c for c in columns_to_drop if c in dataframe.columns]
    return dataframe.drop(columns=existing)

# TODO: Add/remove columns based on your analysis
useless_features = [
    "parcelid",
    "rawcensustractandblock",
    "censustractandblock",
    # "assessmentyear",  # uncomment if all values are the same
    # "propertycountylandusecode",  # uncomment if text codes aren't useful
    # "propertyzoningdesc",  # uncomment if too many unique text values
]

df_dropped_features = drop_useless_features(df, useless_features)
print(f"Shape after dropping useless features: {df_dropped_features.shape}")
print(f"Dropped: {set(df.columns) - set(df_dropped_features.columns)}")

#### **3.A Discussion:** Justify your decisions about which features to drop.

*YOUR ANSWER HERE*

---
## 3.B — Drop Features with Too Many Null Values

In [ ]:
def drop_high_null_features(dataframe: pd.DataFrame, threshold_pct: float = 80.0) -> pd.DataFrame:
    """Drop columns where missing percentage exceeds threshold."""
    pct_missing = (dataframe.isnull().sum() / len(dataframe)) * 100
    cols_to_drop = pct_missing[pct_missing > threshold_pct].index.tolist()
    print(f"Dropping {len(cols_to_drop)} features with >{threshold_pct}% missing: {cols_to_drop}")
    return dataframe.drop(columns=cols_to_drop)

# TODO: Adjust threshold based on your investigation
df_dropped_nulls = drop_high_null_features(df_dropped_features, threshold_pct=80.0)
print(f"Shape after dropping high-null features: {df_dropped_nulls.shape}")

#### **3.B Discussion:** Explain your decision about which features were dropped.

*YOUR ANSWER HERE*

---
## 3.C — Drop Problematic Samples

In [ ]:
def drop_null_target(dataframe: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """Remove rows where the target is null."""
    before = len(dataframe)
    result = dataframe.dropna(subset=[target_col])
    print(f"Dropped {before - len(result)} rows with null target")
    return result

def drop_high_null_rows(dataframe: pd.DataFrame, threshold_pct: float = 50.0) -> pd.DataFrame:
    """Remove rows where the % of null values exceeds the threshold."""
    max_nulls = int(len(dataframe.columns) * (threshold_pct / 100))
    before = len(dataframe)
    result = dataframe.dropna(thresh=len(dataframe.columns) - max_nulls)
    print(f"Dropped {before - len(result)} rows with >{threshold_pct}% null columns")
    return result

def drop_target_outliers(dataframe: pd.DataFrame, target_col: str, z_threshold: float = 3.0) -> pd.DataFrame:
    """Remove rows where the target's z-score exceeds the threshold."""
    z_scores = np.abs((dataframe[target_col] - dataframe[target_col].mean()) / dataframe[target_col].std())
    before = len(dataframe)
    result = dataframe[z_scores < z_threshold]
    print(f"Dropped {before - len(result)} target outlier rows (|z| > {z_threshold})")
    return result

In [ ]:
df_clean = drop_null_target(df_dropped_nulls, TARGET)
df_clean = drop_high_null_rows(df_clean, threshold_pct=50.0)
df_dropped_samples = drop_target_outliers(df_clean, TARGET, z_threshold=3.0)
print(f"Shape after dropping samples: {df_dropped_samples.shape}")

#### **3.C Discussion:** Explain your decisions about which samples were dropped.

*YOUR ANSWER HERE*

---
## 3.D — Impute Remaining Missing Values

Strategy guidance (from Appendix 2):
- **Numerical + normally distributed** → mean
- **Numerical + skewed** → median
- **Categorical** → mode or "Unknown"

In [ ]:
def impute_numerical(dataframe: pd.DataFrame, strategy: str = "median") -> pd.DataFrame:
    """Impute missing values in numerical columns."""
    num_cols = dataframe.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy=strategy)
    dataframe[num_cols] = imputer.fit_transform(dataframe[num_cols])
    return dataframe

def impute_categorical(dataframe: pd.DataFrame, fill_value: str = "Unknown") -> pd.DataFrame:
    """Impute missing values in categorical (object) columns."""
    cat_cols = dataframe.select_dtypes(include=["object", "category"]).columns
    dataframe[cat_cols] = dataframe[cat_cols].fillna(fill_value)
    return dataframe

df_imputed = df_dropped_samples.copy()
df_imputed = impute_numerical(df_imputed, strategy="median")
df_imputed = impute_categorical(df_imputed, fill_value="Unknown")

assert df_imputed.isnull().sum().sum() == 0, "There are still null values!"
print(f"Shape after imputation: {df_imputed.shape}")
print(f"Total remaining nulls: {df_imputed.isnull().sum().sum()}")

#### **3.D Discussion:** Describe your imputation decisions.

*YOUR ANSWER HERE*

---
## 3.E — Encode Categorical Features

In [ ]:
def encode_categoricals(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Ordinal-encode all remaining object columns."""
    cat_cols = dataframe.select_dtypes(include=["object", "category"]).columns.tolist()
    if not cat_cols:
        print("No categorical columns to encode.")
        return dataframe
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    dataframe[cat_cols] = encoder.fit_transform(dataframe[cat_cols])
    print(f"Encoded {len(cat_cols)} categorical columns: {cat_cols}")
    return dataframe

df_encoded = encode_categoricals(df_imputed.copy())
print(f"Final shape: {df_encoded.shape}")
print(f"Dtypes:\n{df_encoded.dtypes.value_counts()}")

---
## Save Cleaned Data for Downstream Notebooks

In [ ]:
df_encoded.to_csv("zillow_cleaned.csv", index=False)
print("Saved zillow_cleaned.csv")

### Next Notebook → `04_feature_relationships.ipynb`